# Recognising Hand Gestures

The **Gesture Recognizer** is a computer vision model that does two jobs at once. First it finds **21 landmarks** on each hand — fingertips, knuckles, and the wrist. Then it looks at the shape those points make and names the gesture.

Codetto uses **MediaPipe's Gesture Recognizer**, which knows eight gestures:

| Gesture | Meaning |
|---|---|
| `None` | No gesture recognised |
| `Closed_Fist` | All fingers curled in |
| `Open_Palm` | All fingers spread, palm forward |
| `Pointing_Up` | Index finger pointing up |
| `Thumb_Down` | Thumb down |
| `Thumb_Up` | Thumb up |
| `Victory` | Index and middle finger in a V |
| `ILoveYou` | The ASL "I love you" sign |

Run the cell below and click **Allow**. Hold a hand up in good light and try a few shapes for about eight seconds.

In [ ]:
from codetto import cv, graphics
import time

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_gesture_detector(camera)

try:
  end = time.time() + 8
  while time.time() < end:
    hands = detector.get_detections()
    canvas.draw_hands(hands)
finally:
  detector.stop()
  camera.stop()

# What a Detection Contains

`detector.get_detections()` returns a list with one dictionary per hand:

| Field | Meaning |
|---|---|
| `gesture` | The gesture name, e.g. `"Thumb_Up"` |
| `confidence` | How sure the model is, `0.0` to `1.0` |
| `handedness` | `"Left"` or `"Right"` |
| `landmarks` | A list of 21 points on that hand |

# Reading Hand Landmarks

Beyond the gesture name, you get the exact position of every joint. You can pick one out with a `cv.HAND` constant, for example `cv.HAND.INDEX_FINGER_TIP` or `cv.HAND.WRIST`. Each landmark is a dictionary with `x`, `y`, and `z`.

The cell below prints the index fingertip and wrist positions once every 30 frames. Run the cell, switch to the console, and move your hand around and watch the numbers shift. Press **Stop** when you are done.

In [ ]:
from codetto import cv, graphics

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_gesture_detector(camera)

frame = 0
try:
  while True:
    hands = detector.get_detections()
    canvas.draw_hands(hands)
    if frame % 30 == 0:
      for hand in hands:
        tip = hand['landmarks'][cv.HAND.INDEX_FINGER_TIP]
        wrist = hand['landmarks'][cv.HAND.WRIST]
        print(f"{hand['handedness']} hand — {hand['gesture']}")
        print(f"  index fingertip: ({tip['x']}, {tip['y']})")
        print(f"  wrist: ({wrist['x']}, {wrist['y']})")
    frame += 1
finally:
  detector.stop()
  camera.stop()

# Gesture Challenge

Pick a **target gesture** from the dropdown, run the cell, switch to the console, and try to make it. The output tells you what it currently sees and whether it matches.

Can you score `MATCH!` on every gesture in the list? Press **Stop** when you are done.

In [ ]:
from codetto import cv, graphics

TARGET = "Thumb_Up" #@param ["Closed_Fist", "Open_Palm", "Pointing_Up", "Thumb_Down", "Thumb_Up", "Victory", "ILoveYou"]

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_gesture_detector(camera)

frame = 0
try:
  while True:
    hands = detector.get_detections()
    canvas.draw_hands(hands)
    if frame % 30 == 0:
      if hands:
        for hand in hands:
          result = 'MATCH!' if hand['gesture'] == TARGET else 'not yet...'
          print(f"{hand['gesture']} ({hand['confidence']:.0%}) - {result}")
      else:
        print('No hand detected - hold your hand up to the camera.')
    frame += 1
finally:
  detector.stop()
  camera.stop()

# Two Hands at Once

The Gesture Recognizer tracks up to two hands by default, so the list can hold two dictionaries. That means you can compare them.

The cell below prints a line to the console whenever **both** hands show the **same** gesture. Try matching a Thumb Up with a friend, or use both of your own hands. Press **Stop** when you are done.

In [ ]:
from codetto import cv, graphics

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_gesture_detector(camera)

frame = 0
try:
  while True:
    hands = detector.get_detections()
    canvas.draw_hands(hands)
    if frame % 30 == 0 and len(hands) == 2:
      first = hands[0]['gesture']
      second = hands[1]['gesture']
      if first == second and first != 'None':
        print('Both hands match:', first)
    frame += 1
finally:
  detector.stop()
  camera.stop()

# Check Your Understanding